# PLN Projeto 3: grafo de conhecimento clínico com modelos de linguagem

BioBERT treinado no MACCROBAT extrai entidades, negação e relações do MultiCaRe; um LLM rotula as relações TREATS e REVEALS. O grafo é comparado com o P1 e o P2.

## 0. Setup e definições comuns

In [ ]:
# !pip install -q pandas pyarrow scikit-learn sklearn-crfsuite networkx matplotlib torch transformers accelerate anthropic python-dotenv

In [ ]:
import os, sys, pathlib
# no Colab: clona (ou atualiza) o repositório e entra na pasta do notebook.
# o chdir fica fora do if: depois de reiniciar o ambiente a pasta já existe, mas o cwd volta para /content
if "google.colab" in sys.modules:
    if not pathlib.Path("/content/mc896").exists():
        !git clone -q https://github.com/4wardAerial/mc896.git /content/mc896
    else:  # força a última versão do GitHub (pull falha se o notebook já gerou arquivos que agora estão no git)
        !git -C /content/mc896 fetch -q origin && git -C /content/mc896 reset -q --hard origin/main
    os.chdir("/content/mc896/project3/pipelines/notebooks")
print(pathlib.Path.cwd())

### 0.1 Caminhos e constantes

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]  # project3/

DATA = ROOT / "data"
RAW = DATA / "raw"
EXTERNAL = DATA / "external"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
MODELS = ROOT / "models"   # não vai para o git

# fontes
MACCROBAT_URL = "https://ndownloader.figshare.com/files/21489405"
MACCROBAT_DIR = RAW / "maccrobat"
MULTICARE_CASES = RAW / "multicare" / "cases.parquet"
MULTICARE_META = RAW / "multicare" / "metadata.parquet"
P2_CASES = ROOT.parent / "project2" / "data" / "external" / "cases.csv"  # os 56 casos do P2

MULTICARE_N = 3000               # casos sorteados do MultiCaRe
MULTICARE_CHARS = (300, 15000)   # tamanho mínimo e máximo do caso

# arquivos entre as seções
DOCS = INTERIM / "docs.csv.gz"
SENTENCES = INTERIM / "sentences.csv"   # não vai para o git, é recriado a partir de docs
ENTITIES_GOLD = INTERIM / "entities_maccrobat.csv.gz"
RELATIONS_GOLD = INTERIM / "relations_maccrobat.csv.gz"
OVERLAP = INTERIM / "overlap_maccrobat_multicare.csv.gz"
ENTITIES_BASELINE = INTERIM / "entities_baseline.csv.gz"
ENTITIES_LLM = INTERIM / "entities_llm.csv.gz"
RELATIONS_LLM = INTERIM / "relations_llm.csv.gz"
ENTITIES_PRED = INTERIM / "entities_pred.csv.gz"
RELATIONS_PRED = INTERIM / "relations_pred.csv.gz"
NODES = PROCESSED / "nodes.csv"
EDGES = PROCESSED / "edges.csv"

# modelos
BIOBERT = "dmis-lab/biobert-base-cased-v1.2"
TOKENIZER_KW = {"do_lower_case": False}  # o tokenizer desse modelo vem em minúsculas por engano
MAX_LEN = 256
SEED = 42
NER_DIR = MODELS / "ner"
NEG_DIR = MODELS / "negacao"
REL_DIR = MODELS / "relacoes"
RETRAIN = False   # False: pula o treino se o modelo já estiver salvo

# LLM
LLM_MODEL = "claude-opus-5-5"
LLM_CACHE = INTERIM / "llm_cache.jsonl"          # respostas já obtidas
LLM_PENDING = INTERIM / "llm_pendentes.jsonl"    # prompts sem resposta (quando não há chave de API)

for d in (RAW, EXTERNAL, INTERIM, PROCESSED, MODELS):
    d.mkdir(parents=True, exist_ok=True)

### 0.2 Formato dos arquivos

Offsets `start`/`end` são de caractere no texto do documento (`end` exclusivo).

In [ ]:
import pandas as pd

COLUMNS = {
    "docs": ["doc_id", "source", "split", "text", "age", "gender"],
    "sentences": ["doc_id", "sent_id", "start", "end", "text"],
    "entities": ["doc_id", "ent_id", "sent_id", "start", "end", "text",
                 "fine_type", "coarse_type", "polarity", "source"],
    "relations": ["doc_id", "head_id", "tail_id", "label", "source", "score"],
    "nodes": ["node_id", "type", "label", "props"],
    "edges": ["source_id", "target_id", "type", "props"],
}
# source dos docs: maccrobat | multicare; split: train | dev | test | p2 | infer
# source das entidades: gold | p1 | p2 | biobert; das relações: gold | llm | biobert | rules

ID_COLUMNS = ["doc_id", "sent_id", "ent_id", "head_id", "tail_id", "node_id", "source_id", "target_id"]


def empty(kind):
    return pd.DataFrame(columns=COLUMNS[kind])


def validate(df, kind):
    missing = [c for c in COLUMNS[kind] if c not in df.columns]
    if missing:
        raise ValueError(f"{kind}: faltam colunas {missing}")
    return df[COLUMNS[kind]]


def save(df, kind, path):
    validate(df, kind).to_csv(path, index=False)


def load(kind, path):
    # ids como texto (os PMIDs do MACCROBAT seriam lidos como número)
    return validate(pd.read_csv(path, dtype={c: str for c in ID_COLUMNS}), kind)

### 0.3 Tipos de entidade e relação

In [ ]:
# tipos do MACCROBAT que o modelo de entidades aprende
NER_TYPES = [
    "Sign_symptom", "Disease_disorder", "Diagnostic_procedure", "Lab_value",
    "Medication", "Therapeutic_procedure", "Dosage", "Frequency",
    "Administration", "Duration", "Biological_structure", "Severity",
    "History", "Family_history", "Age", "Sex",
]

# tipo do MACCROBAT -> tipo do grafo (None: não vira nó)
FINE_TO_COARSE = {
    "Disease_disorder": "Diagnosis",
    "Sign_symptom": "Symptom",
    "Diagnostic_procedure": "Exam",
    "Medication": "Treatment",
    "Therapeutic_procedure": "Treatment",
    "Lab_value": "ExamResult",
    "History": "History",
    "Family_history": "FamilyHistory",
    "Dosage": None, "Frequency": None, "Administration": None,
    "Duration": None, "Biological_structure": None, "Severity": None,
    "Age": None, "Sex": None,
}
P2_TYPES = ["Diagnosis", "Symptom", "Exam", "Treatment"]   # tipos do P1/P2

POLARITY = ["POS", "NEG", "UNCERTAIN"]
POLARITY_MAP = {"NEG": "NEG", "UNCERTAIN": "UNCERTAIN", "MAYBE_POS": "UNCERTAIN",
                "MAYBE_NEG": "UNCERTAIN", "POS": "POS"}

RELATIONS = ["MODIFY", "CAUSE", "TREATS", "REVEALS"]

# relações que cada fonte anota (máscara da perda na seção 6)
ANNOTATED_BY = {
    "gold": {"MODIFY", "CAUSE"},              # MACCROBAT
    "llm": {"CAUSE", "TREATS", "REVEALS"},    # MultiCaRe rotulado pelo LLM
}

# (tipo da cabeça, tipo da cauda) possíveis para cada relação.
# MODIFY e CAUSE: pares com 10+ ocorrências no MACCROBAT (cobrem 98% e 82% dessas relações).
CANDIDATE_PAIRS = {
    "MODIFY": {
        ("Lab_value", "Diagnostic_procedure"), ("Lab_value", "Sign_symptom"), ("Lab_value", "Disease_disorder"),
        ("Lab_value", "Biological_structure"), ("Lab_value", "Therapeutic_procedure"), ("Lab_value", "Medication"),
        ("Biological_structure", "Sign_symptom"), ("Biological_structure", "Diagnostic_procedure"),
        ("Biological_structure", "Disease_disorder"), ("Biological_structure", "Therapeutic_procedure"),
        ("Biological_structure", "Biological_structure"), ("Biological_structure", "Lab_value"),
        ("Severity", "Sign_symptom"), ("Severity", "Disease_disorder"),
        ("Duration", "Sign_symptom"), ("Duration", "Medication"),
        ("Dosage", "Medication"), ("Dosage", "Therapeutic_procedure"), ("Administration", "Medication"),
        ("Frequency", "Medication"), ("Frequency", "Sign_symptom"), ("Frequency", "Diagnostic_procedure"),
        ("Sign_symptom", "Diagnostic_procedure"), ("Sign_symptom", "Sign_symptom"),
        ("Sign_symptom", "Therapeutic_procedure"), ("Sign_symptom", "Disease_disorder"),
        ("Disease_disorder", "Diagnostic_procedure"), ("Disease_disorder", "Disease_disorder"),
        ("Disease_disorder", "Sign_symptom"),
        ("Diagnostic_procedure", "Lab_value"), ("Diagnostic_procedure", "Disease_disorder"),
    },
    "CAUSE": {
        ("Disease_disorder", "Sign_symptom"), ("Disease_disorder", "Disease_disorder"),
        ("Sign_symptom", "Sign_symptom"), ("Sign_symptom", "Disease_disorder"),
        ("Medication", "Sign_symptom"), ("Medication", "Disease_disorder"),
    },
    "TREATS": {
        ("Medication", "Disease_disorder"), ("Medication", "Sign_symptom"),
        ("Therapeutic_procedure", "Disease_disorder"), ("Therapeutic_procedure", "Sign_symptom"),
    },
    "REVEALS": {
        ("Diagnostic_procedure", "Disease_disorder"), ("Diagnostic_procedure", "Sign_symptom"),
    },
}

# aresta Patient -> entidade (mesmos nomes do P2)
PATIENT_EDGE = {
    "Diagnosis": "DIAGNOSED_WITH",
    "Symptom": "HAS_SYMPTOM",
    "Exam": "UNDERWENT_EXAM",
    "Treatment": "TREATED_BY",
    "History": "HAS_HISTORY",
    "FamilyHistory": "FAMILY_HISTORY",
}

### 0.4 Divisão em sentenças

In [ ]:
import re

# quebra em fim de linha ou em . ! ? seguido de espaço e maiúscula
_BOUNDARY = re.compile(r"\n+|(?<=[.!?])\s+(?=[A-Z])")


def split_sentences(doc_id, text):
    out, start = [], 0
    for m in list(_BOUNDARY.finditer(text)) + [None]:
        end = m.start() if m else len(text)
        chunk = text[start:end]
        if chunk.strip():
            lead = len(chunk) - len(chunk.lstrip())
            s, e = start + lead, start + len(chunk.rstrip())
            out.append({"doc_id": doc_id, "sent_id": f"{doc_id}_s{len(out)}",
                        "start": s, "end": e, "text": text[s:e]})
        start = m.end() if m else len(text)
    return out


def sentences_df(docs):
    return pd.DataFrame([r for d in docs.itertuples() for r in split_sentences(d.doc_id, d.text)])


def assign_sentence(entities, sentences):
    # sent_id de cada entidade pela posição do início
    by_doc = {d: g.sort_values("start") for d, g in sentences.groupby("doc_id")}

    def find(row):
        g = by_doc.get(row.doc_id)
        if g is None:
            return None
        hit = g[(g.start <= row.start) & (row.start < g.end)]
        return hit.sent_id.iloc[0] if len(hit) else None

    return entities.apply(find, axis=1)

### 0.5 Métricas

In [ ]:
def _prf(tp, n_pred, n_gold):
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(f, 3),
            "tp": tp, "pred": n_pred, "gold": n_gold}


def entity_prf(pred, gold, type_col="coarse_type", mode="exact"):
    # exact: mesmo span e tipo; overlap: mesmo tipo e spans que se cruzam
    rows = []
    for t in sorted(set(gold[type_col].dropna()) | set(pred[type_col].dropna())):
        g = gold[gold[type_col] == t]
        p = pred[pred[type_col] == t]
        if mode == "exact":
            gk = set(zip(g.doc_id, g.start, g.end))
            tp = sum((d, s, e) in gk for d, s, e in zip(p.doc_id, p.start, p.end))
        else:
            tp = 0
            gd = {d: list(zip(x.start, x.end)) for d, x in g.groupby("doc_id")}
            for d, s, e in zip(p.doc_id, p.start, p.end):
                tp += any(s < ge and gs < e for gs, ge in gd.get(d, []))
        rows.append({"type": t, **_prf(tp, len(p), len(g))})
    df = pd.DataFrame(rows)
    tot = _prf(int(df.tp.sum()), int(df.pred.sum()), int(df.gold.sum()))
    return pd.concat([df, pd.DataFrame([{"type": "micro", **tot}])], ignore_index=True)


def relation_prf(pred, gold):
    # casa (doc_id, head_id, tail_id, label): os dois lados precisam usar as mesmas entidades
    key = ["doc_id", "head_id", "tail_id"]
    rows = []
    for lab in sorted(set(gold.label) | set(pred.label)):
        g = set(map(tuple, gold.loc[gold.label == lab, key].values))
        p = set(map(tuple, pred.loc[pred.label == lab, key].values))
        rows.append({"label": lab, **_prf(len(g & p), len(p), len(g))})
    return pd.DataFrame(rows)

### 0.6 Pares candidatos e marcação de entidades

Negação e relações recebem a sentença com marcadores: `denied [E] fever [/E]`, `[H] insulin [/H] for [T] diabetes [/T]`.

In [ ]:
# entidades do P1/P2 só têm o tipo do grafo: usamos um tipo do MACCROBAT equivalente
COARSE_TO_FINE = {"Diagnosis": "Disease_disorder", "Symptom": "Sign_symptom",
                  "Exam": "Diagnostic_procedure", "Treatment": "Medication"}

PAIR_COLUMNS = ["doc_id", "sent_id", "head_id", "tail_id", "head_type", "tail_type", "head_text", "tail_text",
                "head_start", "head_end", "tail_start", "tail_end", "allowed"]


def ent_type(ents):
    return ents.fine_type.fillna(ents.coarse_type.map(COARSE_TO_FINE))


def candidate_pairs(ents, relations=RELATIONS):
    # pares (cabeça, cauda) da mesma sentença; allowed = relações possíveis pelos tipos
    ents = ents.assign(etype=ent_type(ents)).dropna(subset=["etype", "sent_id"])
    rows = []
    for sid, g in ents.groupby("sent_id"):
        recs = list(g.itertuples(index=False))
        for h in recs:
            for t in recs:
                if h.ent_id == t.ent_id or (h.start < t.end and t.start < h.end):
                    continue
                ok = [r for r in relations if (h.etype, t.etype) in CANDIDATE_PAIRS[r]]
                if ok:
                    rows.append([h.doc_id, sid, h.ent_id, t.ent_id, h.etype, t.etype, h.text, t.text,
                                 h.start, h.end, t.start, t.end, ok])
    return pd.DataFrame(rows, columns=PAIR_COLUMNS)


def mark(text, offset, spans):
    # spans = [(início, fim, abre, fecha)], offsets no documento
    ins = []
    for s, e, a, b in spans:
        ins += [(min(max(s - offset, 0), len(text)), 1, a + " "), (min(max(e - offset, 0), len(text)), 0, " " + b)]
    for p, _, m in sorted(ins, reverse=True):
        text = text[:p] + m + text[p:]
    return text


print(mark("The patient denied fever and nausea.", 100, [(119, 124, "[E]", "[/E]")]))

## 1. Dados: MACCROBAT e MultiCaRe

### 1.1 Baixar o MACCROBAT

In [ ]:
import io, zipfile, urllib.request
import numpy as np

if not any(MACCROBAT_DIR.glob("*.ann")):
    MACCROBAT_DIR.mkdir(parents=True, exist_ok=True)
    # o figshare recusa (403) o User-Agent padrão do urllib
    req = urllib.request.Request(MACCROBAT_URL, headers={"User-Agent": "Mozilla/5.0"})
    data = urllib.request.urlopen(req).read()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        # o zip vem com subpasta (e lixo do macOS): extrai só .txt/.ann direto em MACCROBAT_DIR
        for name in z.namelist():
            base = name.rsplit("/", 1)[-1]
            if "__MACOSX" in name or base.startswith(".") or not base.endswith((".txt", ".ann")):
                continue
            (MACCROBAT_DIR / base).write_bytes(z.read(name))

mac_ids = sorted(p.stem for p in MACCROBAT_DIR.glob("*.ann"))  # nome do arquivo = PMID
print(len(mac_ids), "documentos")
assert len(mac_ids) == 200, "MACCROBAT incompleto: confira o download"

### 1.2 Ler o formato BRAT

`T` vira entidade (span descontínuo fica só com o 1º pedaço), `E` é trocado pela sua entidade gatilho, `R` vira relação e o atributo `POLARITY` vira a coluna `polarity`.

In [ ]:
def read_brat(doc_id, text, ann):
    spans, events, polarity, rels = {}, {}, {}, []
    for line in ann.splitlines():
        if not line.strip() or line[0] not in "TEAR":
            continue
        tag, body, *rest = line.split("\t")
        if tag[0] == "T":
            fine, loc = body.split(" ", 1)
            s, e = map(int, loc.split(";")[0].split())
            spans[tag] = (fine, s, e, rest[0] if rest else "")
        elif tag[0] == "E":
            events[tag] = body.split()[0].split(":")[1]
        elif tag[0] == "A":
            name, target, *val = body.split()
            if name == "POLARITY" and val:
                polarity[target] = val[0]
        elif tag[0] == "R":
            label, a1, a2 = body.split()
            rels.append((label, a1.split(":")[1], a2.split(":")[1]))

    to_t = lambda x: events.get(x, x)  # evento -> entidade
    pol = {to_t(k): POLARITY_MAP.get(v, "POS") for k, v in polarity.items()}

    ents = pd.DataFrame([{
        "doc_id": doc_id, "ent_id": f"{doc_id}_{t}", "sent_id": None,
        "start": s, "end": e, "text": text[s:e],
        "fine_type": fine, "coarse_type": FINE_TO_COARSE.get(fine),
        "polarity": pol.get(t, "POS"), "source": "gold",
        "ann_text": ann_text,  # só para conferir na 1.5
    } for t, (fine, s, e, ann_text) in spans.items()])
    rel = pd.DataFrame([{
        "doc_id": doc_id, "head_id": f"{doc_id}_{to_t(a)}", "tail_id": f"{doc_id}_{to_t(b)}",
        "label": label, "source": "gold", "score": 1.0,
    } for label, a, b in rels])
    return ents, rel


def read_file(path):
    with open(path, encoding="utf-8", newline="") as f:  # newline="" mantém os offsets certos
        return f.read()

### 1.3 MACCROBAT: divisão por documento (70/15/15)

In [ ]:
order = np.random.default_rng(SEED).permutation(mac_ids)
n_tr, n_dev = int(0.70 * len(order)), int(0.15 * len(order))
split_of = {d: ("train" if i < n_tr else "dev" if i < n_tr + n_dev else "test")
            for i, d in enumerate(order)}

docs_mac, ents_mac, rels_mac = [], [], []
for d in mac_ids:
    text = read_file(MACCROBAT_DIR / f"{d}.txt")
    e, r = read_brat(d, text, read_file(MACCROBAT_DIR / f"{d}.ann"))
    docs_mac.append({"doc_id": d, "source": "maccrobat", "split": split_of[d],
                     "text": text, "age": None, "gender": None})
    ents_mac.append(e)
    rels_mac.append(r)

docs_mac = pd.DataFrame(docs_mac)
ents_mac = pd.concat(ents_mac, ignore_index=True)
rels_mac = pd.concat(rels_mac, ignore_index=True)
print(docs_mac.split.value_counts().to_dict(), "|", len(ents_mac), "entidades |", len(rels_mac), "relações")

### 1.4 MultiCaRe

Os 56 casos do P2 entram com o texto do P2 (`split = "p2"`); mais `MULTICARE_N` casos sorteados (`split = "infer"`), sem os artigos que também estão no MACCROBAT.

In [ ]:
if MULTICARE_CASES.exists():
    cases = pd.read_parquet(MULTICARE_CASES).explode("cases")
    cases = pd.json_normalize(cases["cases"].tolist())
    cases["article_id"] = cases.case_id.str.split("_").str[0]

    meta = pd.read_parquet(MULTICARE_META)
    meta = pd.concat([meta[["article_id"]], pd.json_normalize(meta["article_metadata"].tolist())], axis=1)
    overlap = meta.loc[meta.pmid.astype(str).isin(mac_ids), ["article_id", "pmid", "title"]]
    print(len(cases), "casos no MultiCaRe |", len(overlap), "artigos também no MACCROBAT")

    p2 = pd.read_csv(P2_CASES).dropna(subset=["case_text"]).drop_duplicates("case_id")
    docs_p2 = pd.DataFrame({"doc_id": p2.case_id, "source": "multicare", "split": "p2",
                            "text": p2.case_text, "age": p2.age, "gender": p2.gender})

    lo, hi = MULTICARE_CHARS
    pool = cases[~cases.case_id.isin(p2.case_id)
                 & ~cases.article_id.isin(overlap.article_id)
                 & cases.case_text.str.len().between(lo, hi)]
    sample = pool.sample(n=min(MULTICARE_N, len(pool)), random_state=SEED)
    docs_mc = pd.DataFrame({"doc_id": sample.case_id, "source": "multicare", "split": "infer",
                            "text": sample.case_text, "age": sample.age, "gender": sample.gender})
    docs_mc = pd.concat([docs_p2, docs_mc], ignore_index=True)
else:
    print("parquet do MultiCaRe não encontrado: usando os casos de", DOCS.name)
    old = load("docs", DOCS)
    docs_mc = old[old.source == "multicare"]
    overlap = pd.read_csv(OVERLAP)

docs = pd.concat([docs_mac, docs_mc], ignore_index=True)
print(docs.groupby(["source", "split"]).size().to_dict())

### 1.5 Sentenças e conferências

In [ ]:
sentences = sentences_df(docs)
ents_mac["sent_id"] = assign_sentence(ents_mac, sentences)
print(len(sentences), "sentenças")

# texto do .ann x recorte do .txt (só os spans descontínuos devem diferir)
mismatch = ents_mac[ents_mac.text != ents_mac.ann_text]
print("entidades com texto diferente do .ann:", len(mismatch))

s_end = ents_mac.sent_id.map(sentences.set_index("sent_id").end)
print("sem sentença:", ents_mac.sent_id.isna().sum(), "| atravessam a sentença:", (ents_mac.end > s_end).sum())

sent_of = ents_mac.set_index("ent_id").sent_id
same = rels_mac.head_id.map(sent_of) == rels_mac.tail_id.map(sent_of)
print("\n% relações na mesma sentença:")
print(same.groupby(rels_mac.label).agg(["size", "mean"]).sort_values("size", ascending=False).round(2))

print("\nentidades por tipo do grafo:", ents_mac.coarse_type.value_counts().to_dict())
print("polaridade:", ents_mac.polarity.value_counts().to_dict())

In [ ]:
d = docs_mac[docs_mac.split == "train"].doc_id.iloc[0]
print(docs_mac.set_index("doc_id").text[d][:400], "...\n")
print(ents_mac[ents_mac.doc_id == d][["text", "fine_type", "coarse_type", "polarity"]].head(15).to_string())

### 1.6 Salvar

In [ ]:
save(docs, "docs", DOCS)
save(sentences, "sentences", SENTENCES)
save(ents_mac, "entities", ENTITIES_GOLD)
save(rels_mac, "relations", RELATIONS_GOLD)
overlap.to_csv(OVERLAP, index=False)
for p in (DOCS, SENTENCES, ENTITIES_GOLD, RELATIONS_GOLD, OVERLAP):
    print(f"{p.name:35s} {p.stat().st_size / 1e6:6.1f} MB")

## 2. Baselines P1 e P2

Regras e CRF do P2 no teste do MACCROBAT.

### 2.1 Código do P2

In [ ]:
import random
import sklearn_crfsuite

TOKEN_RE = re.compile(r"""
    \d+(?:\.\d+)?
    |[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ0-9]*(?:[-'][A-Za-zÀ-ÿ0-9]+)*
    |[.,;:!?()%/\[\]=<>+-]
""", re.VERBOSE)
STEM_RE = re.compile(r'(?<!s)s$')


def tokenize(sent):
    return [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(sent)]


def stem(tok):
    return STEM_RE.sub('', tok.lower())  # tira o s final (menos em ss)

In [ ]:
def build_gazetteers():
    return {
        'Diagnosis': ['diabetes mellitus', 'hypertension', 'acute pancreatitis', 'chronic pancreatitis', 'pancreatic pseudocyst', 'pseudocyst', 'solid mass', 'type 2 diabetes', 'tumor', 'cancer',
            'acute respiratory distress syndrome', 'sepsis', 'septic shock', 'pneumonia', 'pulmonary embolism', 'acute heart failure', 'congestive heart failure', 'myocardial infarction',
            'ischemic stroke', 'hemorrhagic stroke', 'acute kidney injury', 'chronic kidney disease', 'renal failure', 'acute liver failure', 'cirrhosis', 'hepatitis',
            'cholangitis', 'biliary obstruction', 'appendicitis', 'peritonitis', 'gastrointestinal bleeding', 'intestinal obstruction', 'peptic ulcer disease',
            'deep vein thrombosis', 'venous thromboembolism', 'anemia', 'thrombocytopenia', 'leukemia', 'lymphoma', 'hemolytic anemia', 'fatty liver disease',
            'metastatic disease', 'meningitis', 'encephalitis', 'pyelonephritis', 'urinary tract infection', 'bronchitis', 'asthma', 'pneumonitis', 'respiratory failure',
            'arrhythmia', 'atrial fibrillation', 'valvular disease', 'endocarditis', 'pericarditis', 'liver abscess', 'splenomegaly'
        ],
        'Symptom': ['epigastric pain', 'nausea', 'fever', 'epigastric tenderness', 'fat stranding', 'pain', 'headache', 'cough',
            'abdominal pain', 'vomiting', 'diarrhea', 'constipation', 'bloody stools', 'hematochezia', 'melena', 'shortness of breath', 'dyspnea', 'wheezing',
            'chest pain', 'palpitations', 'dizziness', 'syncope', 'fatigue', 'weakness', 'malaise', 'loss of appetite', 'anorexia', 'weight loss', 'weight gain',
            'swelling', 'edema', 'leg swelling', 'lower extremity edema', 'jaundice', 'dark urine', 'itching', 'pruritus', 'confusion', 'delirium', 'sleep disturbance',
            'insomnia', 'restlessness', 'anxiety', 'depression', 'tremor', 'muscle weakness', 'myalgia', 'arthralgia', 'back pain', 'joint pain', 'neck pain',
            'sore throat', 'hoarseness', 'hemoptysis', 'cyanosis', 'hematuria', 'dysuria', 'polyuria', 'oliguria', 'urinary retention', 'flank pain', 'cold intolerance',
            'heat intolerance', 'blurred vision', 'visual disturbance', 'vertigo', 'paresthesia', 'numbness', 'tingling', 'bruising', 'easy bruising', 'bleeding', 'nosebleed',
            'productive cough', 'nonproductive cough', 'dysphagia', 'odynophagia', 'hiccups', 'tachycardia', 'hypotension', 'hypertension symptoms', 'chills', 'rigors', 'sweating',
            'night sweats', 'burning sensation', 'pressure sensation', 'abdominal distension', 'bloating', 'gastroesophageal reflux', 'heartburn', 'dyspepsia', 'gastrointestinal pain', 'thirst',
            'polyuria', 'excessive thirst', 'hyperglycemia symptoms', 'hypoglycemia symptoms', 'fainting'
        ],
        'Exam': [
          'computed tomography', 'ct', 'thoracic ct', 'endoscopic ultrasound', 'eus', 'ultrasound', 'laboratory tests', 'mri', 'x-ray', 'echocardiogram', 'echocardiography', 'ecg', 'flow cytometry',
          'hemoglobin', 'hematocrit', 'white blood cell count', 'total leukocyte count', 'leucocytosis', 'lymphocyte count', 'platelet count', 'platelet counts', 'mean corpuscular volume',
          'reticulocyte count', 'haptoglobin', 'adamts13 activity', 'c-reactive protein', 'crp', 'erythrocyte sedimentation rate', 'esr', 'creatinine', 'serum creatinine', 'albumin', 'total protein',
          'ferritin', 'lactate dehydrogenase', 'ldh', 'complement c3', 'complement c4', 'c3', 'c4', 'troponin', 'troponin i', 'ck-mb', 'creatine kinase', 'myoglobin', 'ejection fraction', 'lvef',
          'mean gradient', 'nt-probnp', 'heart rate', 'blood pressure', 'mean arterial blood pressure', 'glycosylated hemoglobin', 'hemoglobin a1c', 'hba1c', 'blood sugar', 'immunoglobulin m',
          'immunoglobulin g', 'igm', 'igg', 'igg levels', 'forced vital capacity', 'fvc', 'forced expiratory volume', 'fev1', 'diffusing capacity', 'dlco', 'oxygen saturation', 'respiratory rate',
          'carcinoembryonic antigen', 'cea', 'pd-l1 tps score','hydroxychloroquine level', 'gtt',
      ],
        'Treatment': [
            'intravenous fluids', 'oral rehydration', 'analgesia', 'analgesics', 'pain control',
            'antibiotics', 'antiviral therapy', 'antifungal therapy', 'antiparasitic therapy',
            'anticoagulation', 'antiplatelet therapy', 'heparin', 'warfarin', 'enoxaparin',
            'insulin', 'metformin', 'glucose control', 'blood pressure control', 'antihypertensive therapy',
            'fluid resuscitation', 'electrolyte replacement', 'oxygen therapy', 'ventilation support',
            'corticosteroids', 'prednisone', 'hydrocortisone', 'dexamethasone', 'steroid therapy',
            'immunosuppressive therapy', 'plasma exchange', 'plasmapheresis', 'intravenous immunoglobulin', 'ivig',
            'chemotherapy', 'radiotherapy', 'targeted therapy', 'immunotherapy', 'hormonal therapy',
            'surgery', 'laparotomy', 'laparoscopy', 'resection', 'drainage', 'debridement',
            'endoscopic drainage', 'cystgastrostomy', 'gastrojejunostomy', 'stent placement', 'biliary drainage',
            'dialysis', 'renal replacement therapy', 'liver transplant', 'kidney transplant', 'transplantation',
            'blood transfusion', 'platelet transfusion', 'erythrocyte transfusion',
            'physical therapy', 'rehabilitation', 'respiratory physiotherapy', 'occupational therapy',
            'bronchodilator therapy', 'inhaled beta agonist', 'nebulized therapy', 'smoking cessation',
            'prophylactic antibiotics', 'empiric therapy', 'supportive care', 'palliative care',
            'nutritional support', 'enteral nutrition', 'parenteral nutrition', 'oral nutrition',
            'thrombolysis', 'vasopressor support', 'inotropic support', 'antiemetics', 'anti-inflammatory therapy',
            'antihistamines', 'acid suppression', 'proton pump inhibitor', 'ppi', 'antacids',
            'beta-blocker therapy', 'statin therapy', 'aspirin', 'ibuprofen', 'acetaminophen', 'paracetamol',
            'antidepressant therapy', 'antiepileptic therapy', 'antiarrhythmic therapy', 'diuretic therapy',
            'vaccination', 'immunization', 'monitoring', 'observation', 'watchful waiting', 'conservative management'
        ]
    }

In [ ]:
def build_phrase_index(gaz):
    # termo (tupla de stems) -> (tipo, termo)
    index, max_len = {}, 1
    for etype, kws in gaz.items():
        for kw in kws:
            key = tuple(stem(t[0]) for t in tokenize(kw))
            if key and key not in index:
                index[key] = (etype, kw)
                max_len = max(max_len, len(key))
    return index, max_len


def match_entities(tokens, index, max_len):
    # casamento guloso: maior termo primeiro, da esquerda para a direita
    stems = [stem(t[0]) for t in tokens]
    i, spans = 0, []
    while i < len(tokens):
        for n in range(min(max_len, len(tokens) - i), 0, -1):
            hit = index.get(tuple(stems[i:i + n]))
            if hit:
                spans.append((i, i + n, hit[0], hit[1]))
                i += n
                break
        else:
            i += 1
    return spans


def spans_to_bio(n, spans):
    tags = ['O'] * n
    for s, e, t, *_ in spans:
        tags[s] = f'B-{t}'
        for k in range(s + 1, e):
            tags[k] = f'I-{t}'
    return tags


def bio_to_spans(tags):
    spans, start, typ = [], None, None
    for i, tag in enumerate(list(tags) + ['O']):
        if tag == 'O' or tag.startswith('B-') or tag[2:] != typ:
            if typ is not None:
                spans.append((start, i, typ))
            start, typ = (i, tag[2:]) if tag != 'O' else (None, None)
    return spans

In [ ]:
def tok_feats(tokens, i):
    # configuração "uni" do P2: palavra, stem, afixos, forma e janela de ±2
    W = [t[0].lower() for t in tokens]
    def w(k):
        j = i + k
        return W[j] if 0 <= j < len(W) else ('<BOS>' if j < 0 else '<EOS>')
    raw = tokens[i][0]
    f = {'bias': 1.0, 'w': w(0), 'stem': stem(raw), 'suf2': w(0)[-2:], 'suf3': w(0)[-3:], 'suf4': w(0)[-4:],
         'pre3': w(0)[:3], 'digit': raw.isdigit(), 'title': raw.istitle(), 'upper': raw.isupper(),
         'hyph': '-' in raw, 'first': i == 0}
    for k in (-2, -1, 1, 2):
        f[f'w[{k}]'] = w(k)
    return f


def featurize(sents):
    return [[tok_feats(s['tokens'], i) for i in range(len(s['tokens']))] for s in sents]

### 2.2 Conversão P2 → P3

O P2 usa spans em índices de token; o P3, offsets de caractere no documento.

In [ ]:
def tokenize_sentences(sents):
    out = []
    for r in sents.itertuples():
        toks = tokenize(r.text)
        if toks:
            out.append({"doc_id": r.doc_id, "sent_id": r.sent_id, "offset": r.start,
                        "text": r.text, "tokens": toks})
    return out


def spans_to_entities(sents, spans_per_sent, source):
    rows = []
    for s, spans in zip(sents, spans_per_sent):
        for k, (a, b, typ, *_) in enumerate(spans):
            cs, ce = s["tokens"][a][1], s["tokens"][b - 1][2]
            rows.append({"doc_id": s["doc_id"], "ent_id": f'{s["sent_id"]}_{source}{k}', "sent_id": s["sent_id"],
                         "start": s["offset"] + cs, "end": s["offset"] + ce, "text": s["text"][cs:ce],
                         "fine_type": None, "coarse_type": typ, "polarity": "POS", "source": source})
    return pd.DataFrame(rows, columns=COLUMNS["entities"])

### 2.3 Rodar P1 e P2

In [ ]:
docs = load("docs", DOCS)
sentences = load("sentences", SENTENCES) if SENTENCES.exists() else sentences_df(docs)
if not SENTENCES.exists():
    save(sentences, "sentences", SENTENCES)

GAZ = build_gazetteers()
INDEX, MAXLEN = build_phrase_index(GAZ)

# teste do MACCROBAT + todo o MultiCaRe
test_ids = set(docs[(docs.source == "maccrobat") & (docs.split == "test")].doc_id)
target_ids = test_ids | set(docs[docs.source == "multicare"].doc_id)
target = tokenize_sentences(sentences[sentences.doc_id.isin(target_ids)])

ents_p1 = spans_to_entities(target, [match_entities(s["tokens"], INDEX, MAXLEN) for s in target], "p1")

# CRF com supervisão fraca nos 56 casos do P2
NEG_KEEP = 0.3  # fração de sentenças sem entidade mantidas (igual ao P2)
rng = random.Random(SEED)
train = tokenize_sentences(sentences[sentences.doc_id.isin(docs[docs.split == "p2"].doc_id)])
for s in train:
    s["weak"] = match_entities(s["tokens"], INDEX, MAXLEN)
train = [s for s in train if s["weak"] or rng.random() < NEG_KEEP]
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.05, max_iterations=80, all_possible_transitions=True)
crf.fit(featurize(train), [spans_to_bio(len(s["tokens"]), s["weak"]) for s in train])
ents_p2 = spans_to_entities(target, [bio_to_spans(t) for t in crf.predict(featurize(target))], "p2")

ents_baseline = pd.concat([ents_p1, ents_p2], ignore_index=True)
save(ents_baseline, "entities", ENTITIES_BASELINE)
print(f"CRF treinado em {len(train)} sentenças | entidades: P1 {len(ents_p1)}, P2 {len(ents_p2)}")

### 2.4 Avaliação no teste do MACCROBAT

Casamento exato (mesmo span) e por sobreposição (spans se cruzam).

In [ ]:
gold = load("entities", ENTITIES_GOLD)
gold_test = gold[gold.doc_id.isin(test_ids) & gold.coarse_type.isin(P2_TYPES)]


def compare(preds, gold):
    rows = []
    for name, pred in preds.items():
        for mode in ("exact", "overlap"):
            r = entity_prf(pred, gold, mode=mode).set_index("type")
            rows.append({"modelo": name, "modo": mode,
                         **{t: r.f1.get(t, 0.0) for t in P2_TYPES},
                         "P": r.precision["micro"], "R": r.recall["micro"], "F1": r.f1["micro"]})
    return pd.DataFrame(rows).set_index(["modelo", "modo"])


is_test = lambda df: df[df.doc_id.isin(test_ids)]
results_baselines = compare({"P1 regras": is_test(ents_p1), "P2 CRF": is_test(ents_p2)}, gold_test)
results_baselines.to_csv(PROCESSED / "results_ner_baselines.csv")
results_baselines

In [ ]:
# entidades do ouro que o P1 mais deixa passar
p1_test = is_test(ents_p1)
hit = {(d, s) for d, s0, e0 in zip(p1_test.doc_id, p1_test.start, p1_test.end)
       for _, s, e in gold_test[gold_test.doc_id == d][["start", "end"]].itertuples() if s < e0 and s0 < e}
missed = gold_test[pd.Series([(d, s) not in hit for d, s in zip(gold_test.doc_id, gold_test.start)],
                             index=gold_test.index, dtype=bool)]
missed.assign(text=missed.text.str.lower()).groupby(["coarse_type", "text"]).size() \
      .sort_values(ascending=False).head(20).to_frame("vezes")

## 3. Modelo de entidades (BioBERT)

### 3.1 Preparação dos exemplos

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BIOBERT, **TOKENIZER_KW)

NER_LABELS = ["O"] + [f"{p}-{t}" for t in NER_TYPES for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(NER_LABELS)}
id2label = {i: l for l, i in label2id.items()}


def resolve_nested(ents):
    # tira sobreposições, a entidade menor tem prioridade
    kept = []
    for r in ents.assign(_len=ents.end - ents.start).sort_values("_len").itertuples():
        if all(r.end <= k.start or r.start >= k.end for k in kept):
            kept.append(r)
    return sorted(kept, key=lambda r: r.start)


def encode_sentence(sent, ents):
    enc = tokenizer(sent.text, truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    char_ent = [None] * len(sent.text)   # entidade de cada caractere
    for k, e in enumerate(ents):
        for c in range(max(0, e.start - sent.start), min(len(sent.text), e.end - sent.start)):
            char_ent[c] = k

    labels, first_tok, word_spans = [], [], []
    prev_word, prev_ent = None, None
    for i, (w, (a, b)) in enumerate(zip(enc.word_ids(), enc["offset_mapping"])):
        if w is None:               # [CLS] / [SEP]
            labels.append(-100)
            continue
        if w == prev_word:          # resto da palavra
            labels.append(-100)
            word_spans[-1][1] = b
            continue
        k = next((char_ent[c] for c in range(a, b) if char_ent[c] is not None), None)
        lab = "O" if k is None else ("I-" if k == prev_ent else "B-") + ents[k].fine_type
        labels.append(label2id[lab])
        first_tok.append(i)
        word_spans.append([a, b])
        prev_word, prev_ent = w, k

    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels,
            "doc_id": sent.doc_id, "sent_id": sent.sent_id, "offset": sent.start, "text": sent.text,
            "first_tok": first_tok, "word_spans": word_spans}


def build_examples(sents, ents):
    ents = ents[ents.fine_type.isin(NER_TYPES)]
    by_sent = {s: resolve_nested(g) for s, g in ents.groupby("sent_id")}
    return [encode_sentence(s, by_sent.get(s.sent_id, [])) for s in sents.itertuples()]


def to_features(examples, keys=("input_ids", "attention_mask", "labels")):
    # só o que vai para o modelo
    return [{k: ex[k] for k in keys} for ex in examples]

In [ ]:
docs = load("docs", DOCS)
sentences = load("sentences", SENTENCES) if SENTENCES.exists() else sentences_df(docs)
gold = load("entities", ENTITIES_GOLD)
split_of = docs.set_index("doc_id").split

mac_sents = sentences[sentences.doc_id.isin(docs[docs.source == "maccrobat"].doc_id)]
examples = {sp: build_examples(mac_sents[mac_sents.doc_id.map(split_of) == sp],
                               gold[gold.doc_id.map(split_of) == sp])
            for sp in ("train", "dev", "test")}
print({sp: len(ex) for sp, ex in examples.items()}, "sentenças")

n_trunc = sum(len(tokenizer(s)["input_ids"]) > MAX_LEN for s in mac_sents.text)
print(f"sentenças cortadas (> {MAX_LEN} subtokens): {n_trunc}")

ex = next(e for e in examples["train"] if any(l > 0 for l in e["labels"]))
print(ex["text"][:120])
print([(tokenizer.convert_ids_to_tokens(t), id2label.get(l, "·"))
       for t, l in zip(ex["input_ids"], ex["labels"])][:25])

In [ ]:
def labels_to_entities(ex, word_labels, source):
    # rótulos BIO por palavra -> entidades com offsets no documento
    out, cur = [], None
    for (a, b), lab in zip(ex["word_spans"], word_labels + ["O"]):
        typ = lab[2:] if lab != "O" else None
        if cur and (lab == "O" or lab.startswith("B-") or typ != cur["fine_type"]):
            out.append(cur)
            cur = None
        if typ and cur is None:
            cur = {"fine_type": typ, "s": a, "e": b}
        elif cur:
            cur["e"] = b
    if cur:
        out.append(cur)
    return [{"doc_id": ex["doc_id"], "ent_id": f'{ex["sent_id"]}_{source}{k}', "sent_id": ex["sent_id"],
             "start": ex["offset"] + c["s"], "end": ex["offset"] + c["e"], "text": ex["text"][c["s"]:c["e"]],
             "fine_type": c["fine_type"], "coarse_type": FINE_TO_COARSE.get(c["fine_type"]),
             "polarity": "POS", "source": source} for k, c in enumerate(out)]


# conferência: rótulos do ouro -> entidades tem que dar F1 perto de 1
back = pd.DataFrame([e for ex in examples["test"]
                     for e in labels_to_entities(ex, [id2label[ex["labels"][i]] for i in ex["first_tok"]], "gold")])
gold_ner_test = gold[(gold.doc_id.map(split_of) == "test") & gold.fine_type.isin(NER_TYPES)]
entity_prf(back, gold_ner_test, type_col="fine_type").set_index("type").loc["micro"]

### 3.2 Fine-tuning

In [ ]:
import torch
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)

gold_dev = gold[(gold.doc_id.map(split_of) == "dev") & gold.fine_type.isin(NER_TYPES)]


def ner_metrics(p):
    # F1 de entidade no dev, para escolher a melhor época
    pred = p.predictions.argmax(-1)
    ents = [e for ex, row in zip(examples["dev"], pred)
            for e in labels_to_entities(ex, [id2label[int(row[j])] for j in ex["first_tok"]], "biobert")]
    r = entity_prf(pd.DataFrame(ents, columns=COLUMNS["entities"]), gold_dev, type_col="fine_type")
    r = r.set_index("type").loc["micro"]
    return {"precision": r.precision, "recall": r.recall, "f1": r.f1}


if (NER_DIR / "config.json").exists() and not RETRAIN:
    print("modelo já treinado em", NER_DIR)
    ner_model = AutoModelForTokenClassification.from_pretrained(NER_DIR)
else:
    ner_model = AutoModelForTokenClassification.from_pretrained(BIOBERT, num_labels=len(NER_LABELS), id2label=id2label, label2id=label2id)
    dataCollator = DataCollatorForTokenClassification(tokenizer)
    hyperparameters = TrainingArguments(
        output_dir=MODELS / "ner_checkpoints",
        num_train_epochs=4,
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        seed=SEED,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(
        model=ner_model,
        args=hyperparameters,
        train_dataset=to_features(examples["train"]),
        eval_dataset=to_features(examples["dev"]),
        data_collator=dataCollator,
        compute_metrics=ner_metrics,
    )
    trainer.train()

    ner_model = trainer.model
    trainer.save_model(NER_DIR)
    tokenizer.save_pretrained(NER_DIR)

### 3.3 Avaliação no teste

In [ ]:
def predict_entities(model, examples, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    rows = []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        enc = tokenizer.pad([{"input_ids": e["input_ids"], "attention_mask": e["attention_mask"]} for e in batch],
                            return_tensors="pt").to(device)
        with torch.no_grad():
            pred = model(**enc).logits.argmax(-1).cpu().tolist()
        for ex, p in zip(batch, pred):
            rows += labels_to_entities(ex, [model.config.id2label[p[j]] for j in ex["first_tok"]], "biobert")
    return pd.DataFrame(rows, columns=COLUMNS["entities"])

In [ ]:
ner_model = globals().get("ner_model")
if ner_model is None and (NER_DIR / "config.json").exists():
    ner_model = AutoModelForTokenClassification.from_pretrained(NER_DIR)

if ner_model is None:
    print("Sem modelo: rode a 3.2 primeiro.")
else:
    pred_test = predict_entities(ner_model, examples["test"])

    print("Por tipo do MACCROBAT:")
    display(entity_prf(pred_test, gold_ner_test, type_col="fine_type"))

    print("Por tipo do grafo, com P1 e P2:")
    gold_test = gold_ner_test[gold_ner_test.coarse_type.isin(P2_TYPES)]
    results_ner = pd.concat([pd.read_csv(PROCESSED / "results_ner_baselines.csv", index_col=[0, 1]),
                             compare({"P3 BioBERT": pred_test[pred_test.coarse_type.isin(P2_TYPES)]}, gold_test)])
    results_ner.to_csv(PROCESSED / "results_ner.csv")
    display(results_ner.round(3))

## 4. Modelo de negação (BioBERT)

POS, NEG ou UNCERTAIN para cada entidade.

### 4.1 Preparação dos exemplos

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BIOBERT, **TOKENIZER_KW)
docs = load("docs", DOCS)
sentences = load("sentences", SENTENCES) if SENTENCES.exists() else sentences_df(docs)
gold = load("entities", ENTITIES_GOLD)
split_of = docs.set_index("doc_id").split
mac_sents = sentences[sentences.doc_id.isin(docs[docs.source == "maccrobat"].doc_id)]

NEG_TYPES = ["Sign_symptom", "Disease_disorder", "Medication", "Therapeutic_procedure", "Diagnostic_procedure"]
POL2ID = {p: i for i, p in enumerate(POLARITY)}


def polarity_examples(sents, ents):
    sent = dict(zip(sents.sent_id, zip(sents.start, sents.text)))
    ents = ents[ent_type(ents).isin(NEG_TYPES) & ents.sent_id.isin(sent)]
    out = []
    for e in ents.itertuples(index=False):
        start, text = sent[e.sent_id]
        marked = mark(text, start, [(e.start, e.end, "[E]", "[/E]")])
        enc = tokenizer(marked, truncation=True, max_length=MAX_LEN)
        out.append({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                    "labels": POL2ID.get(e.polarity, 0),
                    "doc_id": e.doc_id, "ent_id": e.ent_id, "sent_start": start, "sent_text": text,
                    "start": e.start, "end": e.end, "coarse_type": e.coarse_type, "text": marked})
    return out


neg_examples = {sp: polarity_examples(mac_sents[mac_sents.doc_id.map(split_of) == sp],
                                      gold[gold.doc_id.map(split_of) == sp])
                for sp in ("train", "dev", "test")}
display(pd.DataFrame({sp: pd.Series([POLARITY[e["labels"]] for e in ex]).value_counts()
                      for sp, ex in neg_examples.items()}))
for pol in ("NEG", "UNCERTAIN"):
    print(pol, "|", next(e["text"] for e in neg_examples["train"] if POLARITY[e["labels"]] == pol))

### 4.2 Fine-tuning

Classes desbalanceadas (~90% POS): entropia cruzada com peso por classe (raiz do inverso da frequência). Melhor época pelo F1 macro no dev.

In [ ]:
from collections import Counter

import torch.nn.functional as F
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

cnt = Counter(e["labels"] for e in neg_examples["train"])
neg_weights = torch.tensor([(len(neg_examples["train"]) / (len(POLARITY) * cnt[i])) ** 0.5 for i in range(len(POLARITY))])
print("peso por classe:", dict(zip(POLARITY, neg_weights.round(decimals=2).tolist())))


def weighted_ce(outputs, labels, num_items_in_batch=None):
    logits = outputs["logits"]
    return F.cross_entropy(logits, labels, weight=neg_weights.to(logits.device))


def neg_metrics(p):
    pred = p.predictions.argmax(-1)
    f1 = f1_score(p.label_ids, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"macro_f1": f1.mean(), **{f"f1_{c}": f for c, f in zip(POLARITY, f1)}}


if (NEG_DIR / "config.json").exists() and not RETRAIN:
    print("modelo já treinado em", NEG_DIR)
    neg_model = AutoModelForSequenceClassification.from_pretrained(NEG_DIR)
else:
    neg_model = AutoModelForSequenceClassification.from_pretrained(
        BIOBERT, num_labels=len(POLARITY), id2label=dict(enumerate(POLARITY)), label2id=POL2ID)
    neg_trainer = Trainer(
        model=neg_model,
        args=TrainingArguments(
            output_dir=MODELS / "negacao_checkpoints", num_train_epochs=3, learning_rate=2e-5,
            per_device_train_batch_size=32, per_device_eval_batch_size=64,
            eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model="macro_f1",
            logging_steps=50, seed=SEED, report_to="none", fp16=torch.cuda.is_available()),
        train_dataset=to_features(neg_examples["train"]),
        eval_dataset=to_features(neg_examples["dev"]),
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=neg_metrics,
        compute_loss_func=weighted_ce,
    )
    neg_trainer.train()

    neg_model = neg_trainer.model
    neg_trainer.save_model(NEG_DIR)
    tokenizer.save_pretrained(NEG_DIR)

### 4.3 Comparação com o NegEx do P2

As regras do P2 só marcam NEG (nunca UNCERTAIN) e não negam exames.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support


def _re(p):
    return re.compile(r"\b(?:" + p + r")\b", re.I)


NEG_PRE = _re(r"no|not|without|denied|denies|denying|negative for|absence of|free of|ruled out|neither|nor")
NEG_POST = _re(r"(?:was|were|is|are) (?:ruled out|negative|absent|not \w+)")
SCOPE_END = {"but", "however", "although", "though", "except", ";", "."}
NEG_WIN = 5


def negex(ex):
    if ex["coarse_type"] == "Exam":
        return "POS"
    toks = tokenize(ex["sent_text"])
    s, e = ex["start"] - ex["sent_start"], ex["end"] - ex["sent_start"]
    a = next((i for i, t in enumerate(toks) if t[2] > s), len(toks))
    b = next((i for i, t in enumerate(toks) if t[1] >= e), len(toks))
    left = [t[0].lower() for t in toks[max(0, a - NEG_WIN):a]]
    right = [t[0].lower() for t in toks[b:b + NEG_WIN]]
    cut = [k for k, w in enumerate(left) if w in SCOPE_END]
    left = left[cut[-1] + 1:] if cut else left
    cut = [k for k, w in enumerate(right) if w in SCOPE_END]
    right = right[:cut[0]] if cut else right
    return "NEG" if NEG_PRE.search(" ".join(left)) or NEG_POST.search(" ".join(right)) else "POS"


def predict_polarity(model, examples, batch_size=64):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    out = []
    for i in range(0, len(examples), batch_size):
        enc = tokenizer.pad([{k: e[k] for k in ("input_ids", "attention_mask")} for e in examples[i:i + batch_size]],
                            return_tensors="pt").to(device)
        with torch.no_grad():
            out += [model.config.id2label[int(k)] for k in model(**enc).logits.argmax(-1).cpu()]
    return out


def polarity_table(name, gold_lab, pred_lab):
    p, r, f, n = precision_recall_fscore_support(gold_lab, pred_lab, labels=POLARITY, zero_division=0)
    return pd.DataFrame({"modelo": name, "classe": POLARITY, "P": p, "R": r, "F1": f, "n_ouro": n})


test_ex = neg_examples["test"]
y_true = [POLARITY[e["labels"]] for e in test_ex]
y_rule = [negex(e) for e in test_ex]
tables = [polarity_table("P2 NegEx", y_true, y_rule)]

neg_model = globals().get("neg_model")
if neg_model is None and (NEG_DIR / "config.json").exists():
    neg_model = AutoModelForSequenceClassification.from_pretrained(NEG_DIR)
if neg_model is None:
    print("Sem modelo: rode a 4.2 primeiro.")
else:
    y_model = predict_polarity(neg_model, test_ex)
    tables.append(polarity_table("P3 BioBERT", y_true, y_model))

results_negation = pd.concat(tables, ignore_index=True)
results_negation.to_csv(PROCESSED / "results_negation.csv", index=False)
display(results_negation.round(3))

In [ ]:
# casos em que regras e modelo discordam
if neg_model is not None:
    diff = pd.DataFrame({"texto": [e["text"] for e in test_ex], "ouro": y_true, "NegEx": y_rule, "BioBERT": y_model})
    diff = diff[diff.NegEx != diff.BioBERT]
    print(len(diff), "discordâncias")
    with pd.option_context("display.max_colwidth", 150):
        display(diff.sample(min(10, len(diff)), random_state=SEED))

## 5. LLM rotula relações

CAUSE, TREATS e REVEALS em pares de entidades. Sem chave de API, os prompts vão para `llm_pendentes.jsonl`.

### 5.1 Pares candidatos

Entidades do MultiCaRe: as do BioBERT (seção 7) se já existirem, senão as do P2. Entidades negadas ficam de fora.

In [ ]:
docs = load("docs", DOCS)
sentences = load("sentences", SENTENCES) if SENTENCES.exists() else sentences_df(docs)
gold = load("entities", ENTITIES_GOLD)
gold_rels = load("relations", RELATIONS_GOLD)
split_of = docs.set_index("doc_id").split
LLM_RELS = sorted(ANNOTATED_BY["llm"])

if ENTITIES_PRED.exists():
    mc_ents = load("entities", ENTITIES_PRED)
    print("entidades do BioBERT")
else:
    mc_ents = load("entities", ENTITIES_BASELINE).query("source == 'p2'")
    print("entidades provisórias do P2")

infer_ids = set(docs[docs.split == "infer"].doc_id)
mc_ents = mc_ents[mc_ents.doc_id.isin(infer_ids) & (mc_ents.polarity != "NEG")]
mc_pairs = candidate_pairs(mc_ents, LLM_RELS)
print(f"{len(mc_pairs)} pares em {mc_pairs.sent_id.nunique()} sentenças de {mc_pairs.doc_id.nunique()} casos")
display(mc_pairs.allowed.explode().value_counts().to_frame("pares"))

### 5.2 Prompt e chamada

Uma chamada por sentença com todos os pares dela; resposta em JSON validado.

In [ ]:
import hashlib
import json
import os
import threading
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import anthropic
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv(ROOT.parent / ".env")
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
LLM_WORKERS = 8

LLM_SYSTEM = """You annotate relations between medical entities in sentences from clinical case reports.
For each numbered pair (HEAD, TAIL), choose which of the pair's options the sentence states. A pair can have zero, one or more labels.
- CAUSE: HEAD causes, triggers or leads to TAIL ("hypokalemia secondary to diuretics": diuretics CAUSE hypokalemia).
- TREATS: HEAD (a medication or therapeutic procedure) is given to treat TAIL (a disease or symptom).
- REVEALS: HEAD (an exam or diagnostic procedure) shows, detects or confirms TAIL.
Rules:
- Label only what the sentence states or clearly implies. Do not add relations from outside medical knowledge.
- Direction matters: the relation goes from HEAD to TAIL. If it only holds the other way, leave the pair empty.
- A negated finding ("CT showed no mass") has no REVEALS.
- evidence: the shortest span of the sentence that supports the labels; empty string when there are no labels.
Return exactly one item per pair, with the pair number."""


class PairLabel(BaseModel):
    pair: int
    labels: list[Literal["CAUSE", "TREATS", "REVEALS"]]
    evidence: str


class SentenceLabels(BaseModel):
    pairs: list[PairLabel]


_lock = threading.Lock()
llm_cache = {}
if LLM_CACHE.exists():
    for line in LLM_CACHE.read_text(encoding="utf-8").splitlines():
        rec = json.loads(line)
        llm_cache[rec["key"]] = rec["result"]
pending = {}
print(len(llm_cache), "respostas no cache | chave de API:", HAS_KEY)


def llm_prompt(sentence, pairs):
    lines = [f'{k}. HEAD: "{p.head_text}" ({p.head_type}) | TAIL: "{p.tail_text}" ({p.tail_type}) | options: {", ".join(p.allowed)}'
             for k, p in enumerate(pairs.itertuples(index=False), 1)]
    return f"Sentence: {sentence}\n\nPairs:\n" + "\n".join(lines)


def llm_call(prompt):
    key = hashlib.sha1((LLM_MODEL + LLM_SYSTEM + prompt).encode()).hexdigest()
    if key in llm_cache:
        return llm_cache[key]
    if not HAS_KEY:
        pending[key] = prompt
        return {"pairs": [], "status": "pendente"}
    client = anthropic.Anthropic(max_retries=5)
    resp = client.beta.messages.parse(
        model=LLM_MODEL,
        max_tokens=4096,
        system=LLM_SYSTEM,
        messages=[{"role": "user", "content": prompt}],
        output_format=SentenceLabels,
        output_config={"effort": "low"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",   # se a sentença for recusada, tenta em outro modelo
    )
    if resp.stop_reason == "refusal" or resp.parsed_output is None:
        result = {"pairs": [], "status": str(resp.stop_reason)}
    else:
        result = {**resp.parsed_output.model_dump(), "status": "ok"}
    with _lock:
        llm_cache[key] = result
        with open(LLM_CACHE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "result": result}, ensure_ascii=False) + "\n")
    return result


def save_pending():
    # prompts sem resposta, para responder fora do notebook
    with open(LLM_PENDING, "w", encoding="utf-8") as f:
        for key, prompt in pending.items():
            f.write(json.dumps({"key": key, "system": LLM_SYSTEM, "prompt": prompt}, ensure_ascii=False) + "\n")
    print(len(pending), "sentenças pendentes em", LLM_PENDING.name)


def run_llm(pairs, sents):
    # uma linha por par com os rótulos do LLM (sentenças pendentes ficam de fora)
    text = dict(zip(sents.sent_id, sents.text))
    groups = [g.reset_index(drop=True) for _, g in pairs.groupby("sent_id")]
    with ThreadPoolExecutor(LLM_WORKERS) as pool:
        results = list(pool.map(lambda g: llm_call(llm_prompt(text[g.sent_id[0]], g)), groups))
    rows = []
    for g, res in zip(groups, results):
        if res["status"] == "pendente":
            continue
        by_num = {p["pair"]: p for p in res["pairs"]}
        for k, p in enumerate(g.itertuples(index=False), 1):
            ans = by_num.get(k, {"labels": [], "evidence": ""})
            labels = [l for l in ans["labels"] if l in p.allowed]
            rows.append({**p._asdict(), "llm_labels": labels, "evidence": ans["evidence"], "status": res["status"]})
    return pd.DataFrame(rows, columns=PAIR_COLUMNS + ["llm_labels", "evidence", "status"])


def llm_to_relations(labeled):
    rel = labeled.explode("llm_labels").dropna(subset=["llm_labels"])
    return pd.DataFrame({"doc_id": rel.doc_id, "head_id": rel.head_id, "tail_id": rel.tail_id,
                         "label": rel.llm_labels, "source": "llm", "score": 1.0}, columns=COLUMNS["relations"])


s0 = mc_pairs.sent_id.iloc[0]
print(llm_prompt(sentences.set_index("sent_id").text[s0], mc_pairs[mc_pairs.sent_id == s0]))

### 5.3 Calibração no MACCROBAT

Mesmo prompt nas entidades gold do dev e do teste; o CAUSE do LLM é comparado com o CAUSE humano.

In [ ]:
cal_ids = set(docs[docs.split.isin(["dev", "test"]) & (docs.source == "maccrobat")].doc_id)
cal_ents = gold[gold.doc_id.isin(cal_ids) & gold.fine_type.isin(NER_TYPES) & (gold.polarity != "NEG")]
cal_pairs = candidate_pairs(cal_ents, LLM_RELS)
cal_pairs = cal_pairs[cal_pairs.sent_id.isin(cal_pairs[cal_pairs.allowed.map(lambda a: "CAUSE" in a)].sent_id)]
print(f"calibração: {cal_pairs.sent_id.nunique()} sentenças, {len(cal_pairs)} pares")

cal_labeled = run_llm(cal_pairs, sentences)
if len(cal_labeled) == 0:
    print("nenhuma resposta ainda")
else:
    done = set(cal_labeled.sent_id)
    cal_pred = llm_to_relations(cal_labeled).query("label == 'CAUSE'")
    ids = set(cal_ents[cal_ents.sent_id.isin(done)].ent_id)
    cal_gold = gold_rels[(gold_rels.label == "CAUSE") & gold_rels.head_id.isin(ids) & gold_rels.tail_id.isin(ids)]
    results_llm_cal = relation_prf(cal_pred, cal_gold)
    results_llm_cal.to_csv(PROCESSED / "results_llm_calibration.csv", index=False)
    print(f"{len(done)} sentenças respondidas")
    display(results_llm_cal)

    gk = set(zip(cal_gold.head_id, cal_gold.tail_id))
    cal_labeled["gold_cause"] = [(h, t) in gk for h, t in zip(cal_labeled.head_id, cal_labeled.tail_id)]
    cal_labeled["llm_cause"] = cal_labeled.llm_labels.map(lambda l: "CAUSE" in l)
    with pd.option_context("display.max_colwidth", 120):
        display(cal_labeled[cal_labeled.gold_cause != cal_labeled.llm_cause]
                [["head_text", "tail_text", "gold_cause", "llm_cause", "evidence"]].head(15))

### 5.4 Rotular o MultiCaRe

Também gera `revisao_llm.csv` (70 pares rotulados + 30 sem rótulo) para a equipe marcar `ok` = 1 ou 0.

In [ ]:
LLM_MAX_SENTENCES = 1000
REVIEW = INTERIM / "revisao_llm.csv"

sids = mc_pairs.sent_id.drop_duplicates()
sids = set(sids.sample(min(LLM_MAX_SENTENCES, len(sids)), random_state=SEED))
labeled = run_llm(mc_pairs[mc_pairs.sent_id.isin(sids)], sentences)
save_pending()   # pendentes da 5.3 e da 5.4

if len(labeled) == 0:
    print("nenhuma resposta ainda")
else:
    rel_llm = llm_to_relations(labeled)
    save(rel_llm, "relations", RELATIONS_LLM)
    save(mc_ents[mc_ents.sent_id.isin(set(labeled.sent_id))], "entities", ENTITIES_LLM)
    print(f"{labeled.sent_id.nunique()} sentenças, {len(labeled)} pares rotulados")
    display(rel_llm.label.value_counts().to_frame("relações do LLM"))

    if not REVIEW.exists():
        text = dict(zip(sentences.sent_id, sentences.text))
        has = labeled.llm_labels.map(len) > 0
        sample = pd.concat([labeled[has].sample(min(70, has.sum()), random_state=SEED),
                            labeled[~has].sample(min(30, (~has).sum()), random_state=SEED)])
        sample = sample.assign(sentence=sample.sent_id.map(text),
                               llm_labels=sample.llm_labels.map(lambda l: ",".join(l) or "NONE"), ok="")
        sample[["sent_id", "sentence", "head_text", "tail_text", "allowed", "llm_labels", "evidence", "ok"]].to_csv(REVIEW, index=False)
        print("amostra para revisão:", REVIEW.name)

In [ ]:
# concordância da equipe com o LLM (depois de preencher a coluna ok)
if REVIEW.exists():
    rev = pd.read_csv(REVIEW).dropna(subset=["ok"])
    if len(rev):
        rev["tem_rotulo"] = rev.llm_labels != "NONE"
        display(rev.groupby("tem_rotulo").ok.agg(["mean", "count"])
                .rename(index={True: "LLM rotulou", False: "LLM deixou vazio"}))
    else:
        print("preencha a coluna ok em", REVIEW.name)

## 6. Modelo de relações (multirrótulo com máscara)

Cada exemplo só é cobrado nas relações que a sua fonte anota.

### 6.1 Preparação dos exemplos

MACCROBAT segue a divisão da seção 1; os casos do LLM são divididos 90/10 em treino e dev.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BIOBERT, **TOKENIZER_KW)
docs = load("docs", DOCS)
sentences = load("sentences", SENTENCES) if SENTENCES.exists() else sentences_df(docs)
gold = load("entities", ENTITIES_GOLD)
gold_rels = load("relations", RELATIONS_GOLD)
split_of = docs.set_index("doc_id").split


def relation_examples(sents, ents, rels, source):
    # source=None na inferência (máscara toda 1)
    pairs = candidate_pairs(ents)
    pos = rels.groupby(["head_id", "tail_id"]).label.apply(set).to_dict() if len(rels) else {}
    sent = dict(zip(sents.sent_id, zip(sents.start, sents.text)))
    mask = [float(source is None or r in ANNOTATED_BY[source]) for r in RELATIONS]
    out = []
    for p in pairs[pairs.sent_id.isin(sent)].itertuples(index=False):
        start, text = sent[p.sent_id]
        marked = mark(text, start, [(p.head_start, p.head_end, "[H]", "[/H]"), (p.tail_start, p.tail_end, "[T]", "[/T]")])
        enc = tokenizer(marked, truncation=True, max_length=MAX_LEN)
        labs = pos.get((p.head_id, p.tail_id), set())
        out.append({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                    "labels": [float(r in labs) for r in RELATIONS], "mask": mask,
                    "doc_id": p.doc_id, "head_id": p.head_id, "tail_id": p.tail_id,
                    "head_type": p.head_type, "tail_type": p.tail_type, "source": source or "infer", "text": marked})
    return out


# MACCROBAT (MODIFY e CAUSE)
g_ents = gold[gold.fine_type.isin(NER_TYPES)]
g_rels = gold_rels[gold_rels.label.isin(RELATIONS)]
rel_examples = {sp: relation_examples(sentences, g_ents[g_ents.doc_id.map(split_of) == sp], g_rels, "gold")
                for sp in ("train", "dev", "test")}

# MultiCaRe rotulado pelo LLM (CAUSE, TREATS e REVEALS)
if RELATIONS_LLM.exists() and ENTITIES_LLM.exists():
    l_ents, l_rels = load("entities", ENTITIES_LLM), load("relations", RELATIONS_LLM)
    l_docs = np.random.default_rng(SEED).permutation(sorted(l_ents.doc_id.unique()))
    l_dev = set(l_docs[:len(l_docs) // 10])
    rel_examples["train"] += relation_examples(sentences, l_ents[~l_ents.doc_id.isin(l_dev)], l_rels, "llm")
    rel_examples["dev"] += relation_examples(sentences, l_ents[l_ents.doc_id.isin(l_dev)], l_rels, "llm")
else:
    print("Sem rótulos do LLM (seção 5): só exemplos do MACCROBAT.")


def rel_stats(examples):
    df = pd.DataFrame([dict(zip(RELATIONS, e["labels"]), source=e["source"]) for e in examples])
    return df.groupby("source").agg(pares=("MODIFY", "size"), **{r: (r, "sum") for r in RELATIONS})


display(pd.concat({sp: rel_stats(ex) for sp, ex in rel_examples.items()}))
ex = next(e for e in rel_examples["train"] if e["labels"][RELATIONS.index("MODIFY")])
print(ex["text"], "|", dict(zip(RELATIONS, ex["labels"])), "| mask", ex["mask"])

### 6.2 Perda com máscara

In [ ]:
def masked_bce(logits, targets, mask, pos_weight=None):
    # logits, targets, mask: [batch, n_relações]
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none", pos_weight=pos_weight)
    return (loss * mask).sum() / mask.sum().clamp(min=1)


def rel_pos_weight(examples):
    # negativos / positivos por relação, só onde a máscara é 1
    y = torch.tensor([e["labels"] for e in examples])
    m = torch.tensor([e["mask"] for e in examples])
    return ((1 - y) * m).sum(0) / (y * m).sum(0).clamp(min=1)


def rel_metrics(p, examples=None, threshold=0.5):
    # F1 por relação no dev, só onde a máscara é 1
    examples = examples or rel_examples["dev"]
    m = np.array([e["mask"] for e in examples]) > 0
    pred = 1 / (1 + np.exp(-p.predictions)) >= threshold
    y = p.label_ids > 0.5
    out = {}
    for k, r in enumerate(RELATIONS):
        tp = (pred[:, k] & y[:, k] & m[:, k]).sum()
        n_pred, n_gold = (pred[:, k] & m[:, k]).sum(), (y[:, k] & m[:, k]).sum()
        out[f"f1_{r}"] = 2 * tp / (n_pred + n_gold) if n_pred + n_gold else 0.0
    out["macro_f1"] = float(np.mean([out[f"f1_{r}"] for r in RELATIONS if m[:, RELATIONS.index(r)].any()]))
    return out


# exemplo: dois pares sem relação, um do MACCROBAT e um do LLM, que o modelo chutou como TREATS e REVEALS
logits = torch.tensor([[-2.0, -2.0, 3.0, 3.0], [-2.0, -2.0, 3.0, 3.0]])
mask = torch.tensor([[1.0, 1.0, 0.0, 0.0], [0.0, 1.0, 1.0, 1.0]])
print("perda sem máscara:", masked_bce(logits, torch.zeros(2, 4), torch.ones(2, 4)).item())
print("perda com máscara:", masked_bce(logits, torch.zeros(2, 4), mask).item())

### 6.3 Fine-tuning

O `Trainer` não sabe usar a máscara: a subclasse tira `labels` e `mask` do lote e calcula a perda. Só uma parte (`REL_NEG_KEEP`) dos pares sem nenhuma relação entra no treino.

In [ ]:
REL_KEYS = ("input_ids", "attention_mask", "labels", "mask")
REL_NEG_KEEP = 0.3


class MaskedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        mask = inputs.pop("mask")
        outputs = model(**inputs)
        loss = masked_bce(outputs.logits, labels, mask, pos_weight.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


if (REL_DIR / "config.json").exists() and not RETRAIN:
    print("modelo já treinado em", REL_DIR)
    rel_model = AutoModelForSequenceClassification.from_pretrained(REL_DIR)
else:
    rng = np.random.default_rng(SEED)
    rel_train = [e for e in rel_examples["train"] if any(e["labels"]) or rng.random() < REL_NEG_KEEP]
    pos_weight = rel_pos_weight(rel_train)
    print(len(rel_train), "exemplos de treino | pos_weight:",
          dict(zip(RELATIONS, pos_weight.round(decimals=1).tolist())))

    rel_model = AutoModelForSequenceClassification.from_pretrained(
        BIOBERT, num_labels=len(RELATIONS), problem_type="multi_label_classification",
        id2label=dict(enumerate(RELATIONS)), label2id={r: i for i, r in enumerate(RELATIONS)})
    rel_trainer = MaskedTrainer(
        model=rel_model,
        args=TrainingArguments(
            output_dir=MODELS / "relacoes_checkpoints", num_train_epochs=3, learning_rate=3e-5,
            per_device_train_batch_size=32, per_device_eval_batch_size=64,
            eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model="macro_f1",
            logging_steps=100, seed=SEED, report_to="none", fp16=torch.cuda.is_available(),
            remove_unused_columns=False),   # senão o Trainer descarta a "mask"
        train_dataset=to_features(rel_train, REL_KEYS),
        eval_dataset=to_features(rel_examples["dev"], REL_KEYS),
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=rel_metrics,
    )
    rel_trainer.train()

    rel_model = rel_trainer.model
    rel_trainer.save_model(REL_DIR)
    tokenizer.save_pretrained(REL_DIR)

### 6.4 Avaliação

Sobre entidades gold. MODIFY e CAUSE contra o teste do MACCROBAT (humano); CAUSE, TREATS e REVEALS contra o dev do LLM. Baseline: gatilhos de relação do P2.

In [ ]:
REL_THRESHOLD = 0.5
MAX_BETWEEN = 10   # máximo de palavras entre as entidades nas regras

CAUSED_BY_RE = _re(r"secondary to|due to|caused by|resulting from|attributed to|because of|as a result of|owing to|induced by")
CAUSES_RE = _re(r"leading to|led to|resulting in|resulted in|results in|causing|caused(?! by)|giving rise to")
REVEALS_FWD = _re(r"showed|shows|showing|revealed|reveals|revealing|demonstrated|demonstrates|demonstrating|confirmed|detected|identified|consistent with|suggestive of|suggesting|evidence of|indicating|indicated|compatible with|notable for|positive for")
REVEALS_BWD = _re(r"confirmed (?:by|on)|diagnosed (?:by|on|via)|seen on|visualized on|detected (?:on|by)|identified (?:on|by)|noted on|demonstrated on|visible on|evident on|found on|shown on|revealed (?:on|by)")
TREATS_FWD = _re(r"for|to treat|against|treatment of|management of|treating|to manage|to control")
TREATS_BWD = _re(r"treated (?:with|by|using|conservatively with)|managed (?:with|by|using|conservatively with)|controlled (?:with|by)|responded to|resolved (?:with|after)|started on|initiated on|given|received|administered|underwent|undergoing|prescribed|therapy with|treatment with")
# (gatilho com a cabeça antes da cauda, gatilho com a cauda antes da cabeça)
RULES = {"CAUSE": (CAUSES_RE, CAUSED_BY_RE), "TREATS": (TREATS_FWD, TREATS_BWD), "REVEALS": (REVEALS_FWD, REVEALS_BWD)}


def rule_relations(pairs, sents):
    sent = dict(zip(sents.sent_id, zip(sents.start, sents.text)))
    rows = []
    for p in pairs.itertuples(index=False):
        start, text = sent[p.sent_id]
        fwd = p.head_end <= p.tail_start
        a, b = (p.head_end, p.tail_start) if fwd else (p.tail_end, p.head_start)
        between = text[a - start:b - start]
        if ";" in between or len(between.split()) > MAX_BETWEEN:
            continue
        for r in p.allowed:
            if r in RULES and RULES[r][0 if fwd else 1].search(between):
                rows.append({"doc_id": p.doc_id, "head_id": p.head_id, "tail_id": p.tail_id,
                             "label": r, "source": "rules", "score": 1.0})
    return pd.DataFrame(rows, columns=COLUMNS["relations"])


def predict_relations(model, examples, batch_size=64, threshold=REL_THRESHOLD):
    # fica a relação acima do limiar e permitida pelos tipos do par
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    rows = []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        enc = tokenizer.pad([{k: e[k] for k in ("input_ids", "attention_mask")} for e in batch], return_tensors="pt").to(device)
        with torch.no_grad():
            prob = torch.sigmoid(model(**enc).logits).cpu().numpy()
        for e, pr in zip(batch, prob):
            for k, r in enumerate(RELATIONS):
                if pr[k] >= threshold and (e["head_type"], e["tail_type"]) in CANDIDATE_PAIRS[r]:
                    rows.append({"doc_id": e["doc_id"], "head_id": e["head_id"], "tail_id": e["tail_id"],
                                 "label": r, "source": "biobert", "score": float(pr[k])})
    return pd.DataFrame(rows, columns=COLUMNS["relations"])


def gold_of(examples):
    return pd.DataFrame([{"doc_id": e["doc_id"], "head_id": e["head_id"], "tail_id": e["tail_id"], "label": r,
                          "source": e["source"], "score": 1.0}
                         for e in examples for r, y in zip(RELATIONS, e["labels"]) if y], columns=COLUMNS["relations"])


def rel_table(name, pred, gold_df, labels, ouro):
    t = relation_prf(pred[pred.label.isin(labels)], gold_df[gold_df.label.isin(labels)])
    return t[t.label.isin(labels)].assign(modelo=name, ouro=ouro)


# ouro do teste: todas as MODIFY/CAUSE entre entidades do teste (as que cruzam sentenças contam como erro)
test_ents = g_ents[g_ents.doc_id.map(split_of) == "test"]
test_gold = g_rels[g_rels.head_id.isin(test_ents.ent_id) & g_rels.tail_id.isin(test_ents.ent_id)]
test_pairs = candidate_pairs(test_ents)
llm_dev = [e for e in rel_examples["dev"] if e["source"] == "llm"]

tables = [rel_table("P2 regras", rule_relations(test_pairs, sentences), test_gold, ["CAUSE"], "humano")]
if llm_dev:
    dev_pairs = candidate_pairs(l_ents[l_ents.doc_id.isin(l_dev)], LLM_RELS)
    tables.append(rel_table("P2 regras", rule_relations(dev_pairs, sentences), gold_of(llm_dev), LLM_RELS, "LLM"))

rel_model = globals().get("rel_model")
if rel_model is None and (REL_DIR / "config.json").exists():
    rel_model = AutoModelForSequenceClassification.from_pretrained(REL_DIR)
if rel_model is None:
    print("Sem modelo: rode a 6.3 primeiro.")
else:
    tables.append(rel_table("P3 BioBERT", predict_relations(rel_model, rel_examples["test"]), test_gold,
                            ["MODIFY", "CAUSE"], "humano"))
    if llm_dev:
        tables.append(rel_table("P3 BioBERT", predict_relations(rel_model, llm_dev), gold_of(llm_dev), LLM_RELS, "LLM"))

results_relations = pd.concat(tables, ignore_index=True)[["ouro", "modelo", "label", "precision", "recall", "f1", "tp", "pred", "gold"]]
results_relations.to_csv(PROCESSED / "results_relations.csv", index=False)
results_relations

## 7. Inferência no MultiCaRe

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BIOBERT, **TOKENIZER_KW)
docs = load("docs", DOCS)
sentences = load("sentences", SENTENCES) if SENTENCES.exists() else sentences_df(docs)
mc_sents = sentences[sentences.doc_id.isin(docs[docs.source == "multicare"].doc_id)]
print(len(mc_sents), "sentenças do MultiCaRe")

if not (NER_DIR / "config.json").exists():
    raise SystemExit("Sem o modelo de entidades: rode a 3.2.")
ner_model = AutoModelForTokenClassification.from_pretrained(NER_DIR)
ents_pred = predict_entities(ner_model, build_examples(mc_sents, empty("entities")))
print(len(ents_pred), "entidades")

if (NEG_DIR / "config.json").exists():
    neg_model = AutoModelForSequenceClassification.from_pretrained(NEG_DIR)
    ex = polarity_examples(mc_sents, ents_pred)
    pol = dict(zip([e["ent_id"] for e in ex], predict_polarity(neg_model, ex)))
    ents_pred["polarity"] = ents_pred.ent_id.map(pol).fillna("POS")
else:
    print("Sem modelo de negação: todas as entidades ficam POS.")
display(ents_pred.groupby(["fine_type", "polarity"]).size().unstack(fill_value=0))
save(ents_pred, "entities", ENTITIES_PRED)

# entidades negadas não entram nas relações
if (REL_DIR / "config.json").exists():
    rel_model = AutoModelForSequenceClassification.from_pretrained(REL_DIR)
    ex = relation_examples(mc_sents, ents_pred[ents_pred.polarity != "NEG"], empty("relations"), None)
    rels_pred = predict_relations(rel_model, ex)
    print(len(ex), "pares candidatos |", rels_pred.label.value_counts().to_dict())
else:
    rels_pred = empty("relations")
    print("Sem modelo de relações: relations_pred vazio.")
save(rels_pred, "relations", RELATIONS_PRED)

## 8. Grafo

Pacientes ligados às entidades; CAUSE, TREATS e REVEALS ligam entidades entre si. Negadas viram `NEG_<tipo>`; valores de exame viram `ExamResult`.

In [ ]:
from collections import defaultdict


def norm_label(text):
    return " ".join(stem(w) for w in re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", str(text).lower()))


def build_graph(docs, ents, rels):
    nodes, surface = {}, defaultdict(Counter)
    edges = defaultdict(lambda: {"count": 0, "cases": set(), "props": defaultdict(set)})

    for d in docs.itertuples(index=False):
        nodes[f"P:{d.doc_id}"] = {"node_id": f"P:{d.doc_id}", "type": "Patient", "label": d.doc_id,
                                  "props": {"age": d.age, "gender": d.gender, "split": d.split}}

    # entidades -> nós e arestas do paciente
    ents = ents[ents.doc_id.isin(set(docs.doc_id))]
    node_of, patient_edge = {}, {}
    for e in ents.itertuples(index=False):
        if e.coarse_type not in PATIENT_EDGE:
            continue
        nid = f"{e.coarse_type}:{norm_label(e.text)}"
        nodes.setdefault(nid, {"node_id": nid, "type": e.coarse_type, "label": None, "props": {}})
        surface[nid][str(e.text).lower()] += 1
        rel = PATIENT_EDGE[e.coarse_type]
        key = (f"P:{e.doc_id}", nid, rel if e.polarity != "NEG" else "NEG_" + rel)
        edges[key]["count"] += 1
        edges[key]["cases"].add(e.doc_id)
        if e.polarity == "UNCERTAIN":
            edges[key]["props"]["uncertain"].add(True)
        node_of[e.ent_id] = (nid, e.polarity)
        patient_edge[e.ent_id] = key

    # relações
    info = ents.set_index("ent_id")[["doc_id", "fine_type", "text"]].to_dict("index")
    for r in rels.itertuples(index=False):
        h, t = info.get(r.head_id), info.get(r.tail_id)
        if h is None or t is None:
            continue
        if r.label == "MODIFY":
            if r.tail_id not in patient_edge:
                continue
            if h["fine_type"] == "Lab_value" and node_of[r.tail_id][0].startswith("Exam:"):
                rid = f"R:{h['doc_id']}:{node_of[r.tail_id][0]}:{norm_label(h['text'])}"
                nodes.setdefault(rid, {"node_id": rid, "type": "ExamResult", "label": h["text"], "props": {}})
                key = (node_of[r.tail_id][0], rid, "HAS_RESULT")
                edges[key]["count"] += 1
                edges[key]["cases"].add(h["doc_id"])
            elif h["fine_type"]:
                edges[patient_edge[r.tail_id]]["props"][h["fine_type"].lower()].add(str(h["text"]).lower())
        elif r.head_id in node_of and r.tail_id in node_of:
            (hn, hp), (tn, tp) = node_of[r.head_id], node_of[r.tail_id]
            if "NEG" in (hp, tp) or hn == tn:
                continue
            key = (hn, tn, r.label)
            edges[key]["count"] += 1
            edges[key]["cases"].add(h["doc_id"])

    for nid, c in surface.items():
        nodes[nid]["label"] = c.most_common(1)[0][0]
    nodes_df = pd.DataFrame([{**n, "props": json.dumps(n["props"], default=str)} for n in nodes.values()],
                            columns=COLUMNS["nodes"])
    edges_df = pd.DataFrame([{"source_id": s, "target_id": t, "type": rel,
                              "props": json.dumps({"count": v["count"], "n_cases": len(v["cases"]),
                                                   **{k: sorted(x) for k, x in v["props"].items()}}, default=str)}
                             for (s, t, rel), v in edges.items()], columns=COLUMNS["edges"])
    return nodes_df, edges_df


def graph_stats(nodes, edges):
    return pd.concat([nodes.type.value_counts().add_prefix("nós "), edges.type.value_counts().add_prefix("arestas ")])

In [ ]:
docs = load("docs", DOCS)
mc_docs = docs[docs.source == "multicare"]

if ENTITIES_PRED.exists():
    g_ents, g_rels = load("entities", ENTITIES_PRED), load("relations", RELATIONS_PRED)
else:
    print("Sem entities_pred (seção 7): grafo provisório com as entidades do P2.")
    g_ents, g_rels = load("entities", ENTITIES_BASELINE).query("source == 'p2'"), empty("relations")

nodes, edges = build_graph(mc_docs, g_ents, g_rels)
save(nodes, "nodes", NODES)
save(edges, "edges", EDGES)
display(graph_stats(nodes, edges).to_frame("P3"))

# relações entre entidades que aparecem em mais casos
ee = edges[~edges.source_id.str.startswith("P:") & (edges.type != "HAS_RESULT")].copy()
if len(ee):
    ee["n_cases"] = ee.props.map(lambda p: json.loads(p)["n_cases"])
    label = nodes.set_index("node_id").label
    display(ee.assign(origem=ee.source_id.map(label), destino=ee.target_id.map(label))
            .nlargest(15, "n_cases")[["origem", "type", "destino", "n_cases"]])

## 9. Avaliação final: P1 × P2 × P3

In [ ]:
def show(title, path, section):
    print(f"--- {title} ---")
    if Path(path).exists():
        display(pd.read_csv(path).round(3))
    else:
        print(f"falta rodar a seção {section}")


ner_path = PROCESSED / "results_ner.csv"
show("Entidades (teste MACCROBAT)", ner_path if ner_path.exists() else PROCESSED / "results_ner_baselines.csv", "3.3")
show("Negação (teste MACCROBAT)", PROCESSED / "results_negation.csv", "4.3")
show("Relações", PROCESSED / "results_relations.csv", "6.4")
show("LLM x humanos no CAUSE", PROCESSED / "results_llm_calibration.csv", "5.3")

In [ ]:
# grafos do P1, P2 e P3 nos mesmos casos do MultiCaRe
docs = load("docs", DOCS)
mc_docs = docs[docs.source == "multicare"]
base = load("entities", ENTITIES_BASELINE)
graphs = {"P1 regras": build_graph(mc_docs, base[base.source == "p1"], empty("relations")),
          "P2 CRF": build_graph(mc_docs, base[base.source == "p2"], empty("relations"))}
if ENTITIES_PRED.exists():
    graphs["P3 BioBERT"] = build_graph(mc_docs, load("entities", ENTITIES_PRED), load("relations", RELATIONS_PRED))

graph_cmp = pd.DataFrame({name: graph_stats(*g) for name, g in graphs.items()}).fillna(0).astype(int)
graph_cmp.to_csv(PROCESSED / "results_graphs.csv")
graph_cmp

### Aplicação

TODO: avaliação da aplicação sobre os três grafos.